# CropCare AI: PlantVillage tomato MobileNetV3Small

This notebook runs the repository training pipeline on the PlantVillage tomato classes. It checks the download, prepares the color images, trains MobileNetV3Small, writes the test-set report, exports float32 TensorFlow Lite, and loads that file for one image.

This file does not contain a trained model or test-set scores. Those files appear only after every code cell below has run.

Class folders, the 70/15/15 group split, and the licence come from `config/classes.plantvillage.tomato.yaml`. That file is the tomato subset of `config/classes.plantvillage.yaml`. Training settings come from `config/train.tomato.yaml`.

PlantVillage images are released under CC BY-SA 3.0. Cite Hughes and Salathé, arXiv:1511.08060 (2015), and Mohanty, Hughes, and Salathé, Frontiers in Plant Science, 2016, doi:10.3389/fpls.2016.01419. The notebook stops when the downloaded dataset card does not state that licence.

Use a Colab GPU runtime. TensorFlow uses the GPU when the runtime provides one.


## 1. Get this version of the repository

Colab opens this notebook by itself. The training package has to be on the VM as well.

Set `CROPCARE_GIT_URL` only when that remote already contains `ml/config/train.tomato.yaml`. Otherwise leave the URL empty and upload a zip of the local `CropDiseaseDetection` repository when the next cell asks. The zip must contain `ml/src/cropcare_ml` and `ml/config/train.tomato.yaml`.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

CROPCARE_GIT_URL = ""
IN_COLAB = "google.colab" in sys.modules


def match_ml(path: Path) -> Path | None:
    package = path / "src" / "cropcare_ml" / "prepare.py"
    config = path / "config" / "train.tomato.yaml"
    if package.is_file() and config.is_file():
        return path
    nested = path / "ml"
    if (nested / "src" / "cropcare_ml" / "prepare.py").is_file() and (nested / "config" / "train.tomato.yaml").is_file():
        return nested
    return None


def find_ml_root(extra_roots: list[Path]) -> Path | None:
    for seed in [Path.cwd(), Path.cwd().parent]:
        found = match_ml(seed)
        if found is not None:
            return found
    for root in extra_roots:
        found = match_ml(root)
        if found is not None:
            return found
        if not root.exists() or not root.is_dir():
            continue
        for config in root.rglob("train.tomato.yaml"):
            if config.parent.name != "config":
                continue
            found = match_ml(config.parent.parent)
            if found is not None:
                return found
    return None


ml_root = find_ml_root([])
if ml_root is None and IN_COLAB and CROPCARE_GIT_URL:
    checkout = Path("/content/CropCare")
    if not (checkout / ".git").exists():
        subprocess.run(["git", "clone", "--depth", "1", CROPCARE_GIT_URL, str(checkout)], check=True)
    ml_root = find_ml_root([checkout])
    if ml_root is None:
        raise SystemExit(
            "The cloned repository has no ml/config/train.tomato.yaml. "
            "Push this commit before cloning, or upload a zip of the local repository."
        )

if ml_root is None and IN_COLAB:
    from google.colab import files

    print("Upload a zip of the CropDiseaseDetection repository.")
    uploaded = files.upload()
    extract_root = Path("/content/uploaded_cropcare")
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True)
    for name, payload in uploaded.items():
        archive = extract_root / name
        archive.write_bytes(payload)
        shutil.unpack_archive(archive, extract_root / "unpacked")
    ml_root = find_ml_root([extract_root])

if ml_root is None:
    raise SystemExit(
        "Could not find ml/config/train.tomato.yaml. Run this notebook from ml/, "
        "or in Colab set CROPCARE_GIT_URL or upload a repository zip."
    )

os.chdir(ml_root)
os.environ["PYTHONPATH"] = str(ml_root / "src")
if str(ml_root / "src") not in sys.path:
    sys.path.insert(0, str(ml_root / "src"))
print(ml_root)


## 2. Install the training dependencies

`requirements-train.txt` installs the base numeric packages and TensorFlow `>=2.16,<2.21`. Restart the runtime and run from the top if Colab reports that a restart is required.


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-train.txt"], check=True)


In [ ]:
import importlib.metadata
import sys

import tensorflow as tf

installed = importlib.metadata.version("tensorflow")
parts = tuple(int(piece) for piece in installed.split(".")[:2])
if parts < (2, 16) or parts >= (2, 21):
    raise SystemExit(f"TensorFlow {installed} is outside the training range >=2.16,<2.21.")
if tf.__version__ != installed:
    raise SystemExit(
        f"This kernel loaded TensorFlow {tf.__version__}, while the installed wheel is {installed}. "
        "Restart the runtime and run every cell again from the top."
    )
print(sys.version)
print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))


## 3. Check the PlantVillage licence

The class names match the `spMohanty/PlantVillage-Dataset` color folders and the `mohanty/PlantVillage` dataset card. The card is the licence text this experiment checks. A public-domain statement in that text stops the notebook.


In [ ]:
import subprocess
import sys
import urllib.request
from pathlib import Path

licence_dir = Path("data/licence")
licence_dir.mkdir(parents=True, exist_ok=True)
card_path = licence_dir / "mohanty_PlantVillage_README.md"
urllib.request.urlretrieve(
    "https://huggingface.co/datasets/mohanty/PlantVillage/raw/main/README.md",
    card_path,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "cropcare_ml.verify",
        "licence",
        "--config",
        "config/classes.plantvillage.tomato.yaml",
        "--text-file",
        str(card_path),
        "--output",
        str(licence_dir / "licence_check.json"),
    ],
    check=True,
)


## 4. Download the mapped tomato color folders

The checkout is limited to the folder names in `config/classes.plantvillage.tomato.yaml`. Grayscale and segmented copies are left out. The Hugging Face 80/20 split is left unused. `cropcare_ml.prepare` assigns whole leaf groups to train, validation, and test with seed 42.


In [ ]:
from pathlib import Path
import subprocess
import sys

from cropcare_ml.config import load_experiment

DATASET_GIT_URL = "https://github.com/spMohanty/PlantVillage-Dataset.git"
experiment = load_experiment(Path("config/classes.plantvillage.tomato.yaml"))
folder_paths = [f"raw/color/{spec.folder_names[0]}" for spec in experiment.classes]
dataset_root = Path("/content/PlantVillage-Dataset") if "google.colab" in sys.modules else Path("data/raw/plantvillage_github").resolve()


def ready(root: Path) -> bool:
    return all((root / folder).is_dir() for folder in folder_paths)


if ready(dataset_root):
    print(f"Reusing {dataset_root}")
elif (dataset_root / ".git").exists():
    subprocess.run(["git", "sparse-checkout", "set", "--no-cone", "README.md", *folder_paths], cwd=dataset_root, check=True)
else:
    if dataset_root.exists():
        raise SystemExit(f"{dataset_root} exists and is not a PlantVillage git checkout.")
    dataset_root.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", DATASET_GIT_URL, str(dataset_root)],
        check=True,
    )
    subprocess.run(["git", "sparse-checkout", "set", "--no-cone", "README.md", *folder_paths], cwd=dataset_root, check=True)

readme = (dataset_root / "README.md").read_text(encoding="utf-8")
if "10.3389/fpls.2016.01419" not in readme:
    raise SystemExit("The cloned README does not cite doi:10.3389/fpls.2016.01419.")
print(dataset_root)


## 5. Count the class folders, then prepare and check the split

The layout check counts files in every mapped tomato folder. Preparation then decodes the images, groups leaf copies, and writes the manifest. The prepared check compares those counts with the manifest, confirms the CC BY-SA 3.0 record, and rejects a leaf group or file hash that lands in two splits. Every mapped tomato class has to be included.


In [ ]:
from pathlib import Path
import subprocess
import sys

from cropcare_ml.config import load_train_config

train_config = load_train_config(Path("config/train.tomato.yaml"))
subprocess.run(
    [
        sys.executable,
        "-m",
        "cropcare_ml.verify",
        "layout",
        "--config",
        "config/classes.plantvillage.tomato.yaml",
        "--data-root",
        str(dataset_root),
        "--require-complete",
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "cropcare_ml.prepare",
        "--config",
        "config/classes.plantvillage.tomato.yaml",
        "--data-root",
        str(dataset_root),
        "--output-dir",
        str(train_config.prepared_dir),
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "cropcare_ml.verify",
        "prepared",
        "--config",
        "config/classes.plantvillage.tomato.yaml",
        "--prepared-dir",
        str(train_config.prepared_dir),
        "--require-all-configured",
    ],
    check=True,
)


## 6. Train MobileNetV3Small and read the test-set report

`cropcare_ml.train` freezes the ImageNet backbone, trains the classifier, fine-tunes the last 20 layers, and scores the test split once. It writes `metrics.json` and `confusion_matrix.csv` from those predictions, then exports `model.tflite`, `labels.json`, and `preprocessing.json`.


In [ ]:
import json
from pathlib import Path

from cropcare_ml.config import load_preprocessing_spec
from cropcare_ml.train import train

load_preprocessing_spec()
run_dir = train(train_config)
metrics = json.loads((run_dir / "metrics.json").read_text(encoding="utf-8"))
print(run_dir)
print(f"Test images: {metrics['sample_count']}")
print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"Macro-F1: {metrics['macro_f1']:.4f}")
for row in metrics["per_class"]:
    print(
        f"{row['code']}: precision {row['precision']:.4f}, "
        f"recall {row['recall']:.4f}, f1 {row['f1']:.4f}, support {row['support']}"
    )
print((run_dir / "confusion_matrix.csv").read_text(encoding="utf-8"))
print((run_dir / "export_record.json").read_text(encoding="utf-8"))


## 7. Load the exported model

The smoke test resizes one test-split image with the same bilinear RGB path as training and runs `model.tflite`. The printed class is a load check. The test-set scores remain the values in `metrics.json`.


In [ ]:
import json
import subprocess
import sys

import tensorflow as tf

from cropcare_ml.manifest import read_jsonl

test_rows = [row for row in read_jsonl(train_config.prepared_dir / "manifest.jsonl") if row["split"] == "test"]
if not test_rows:
    raise SystemExit("The manifest has no test image for the smoke test.")
(run_dir / "environment.json").write_text(
    json.dumps(
        {
            "python": sys.version,
            "tensorflow": tf.__version__,
            "gpus": [device.name for device in tf.config.list_physical_devices("GPU")],
        },
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "cropcare_ml.smoke",
        "--model",
        str(run_dir / "model.tflite"),
        "--labels",
        str(run_dir / "labels.json"),
        "--preprocessing",
        str(run_dir / "preprocessing.json"),
        "--image",
        test_rows[0]["path"],
        "--output-dir",
        str(run_dir),
    ],
    check=True,
)
print((run_dir / "smoke_test.json").read_text(encoding="utf-8"))


## 8. Download the Sprint 3 bundle

The zip contains the model, the label file, the preprocessing file, the test-set report, and the checks from this run. It leaves the images on the Colab VM. Copy the unzipped folder to `ml/artifacts/runs/plantvillage_tomato_colab/` on the development machine. That directory is gitignored. Leave the model and the dataset uncommitted.

```powershell
cd ml
Expand-Archive -Path "$env:USERPROFILE\Downloads\cropcare_plantvillage_tomato_sprint3.zip" -DestinationPath artifacts\runs\plantvillage_tomato_colab -Force
git check-ignore -v artifacts\runs\plantvillage_tomato_colab\model.tflite
```

Sprint 3 reads `model.tflite`, `labels.json`, and `preprocessing.json` from that folder. `metrics.json` and `confusion_matrix.csv` stay beside them as the test-set report.


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

bundle = Path("/content/cropcare_plantvillage_tomato_sprint3") if "google.colab" in sys.modules else Path("artifacts/sprint3_bundle")
if bundle.exists():
    shutil.rmtree(bundle)
subprocess.run(
    [
        sys.executable,
        "-m",
        "cropcare_ml.verify",
        "bundle",
        "--run-dir",
        str(run_dir),
        "--prepared-dir",
        str(train_config.prepared_dir),
        "--destination",
        str(bundle),
        "--licence-check",
        "data/licence/licence_check.json",
    ],
    check=True,
)
archive_base = bundle.parent / "cropcare_plantvillage_tomato_sprint3"
archive = Path(shutil.make_archive(str(archive_base), "zip", bundle))
print(archive)
if "google.colab" in sys.modules:
    from google.colab import files

    files.download(str(archive))
